# Simple Jakarta Air Quality Preprocessing

This notebook combines PM2.5 data from OpenAQ and PurpleAir with weather data
from Meteostat. The main preprocessing steps are:

- convert timestamps to UTC;
- remove duplicates and impossible values;
- calculate hourly averages;
- fill only small gaps using interpolation;
- match each air-quality sensor to its nearest weather station;
- remove highly correlated weather columns;
- save one clean DataFrame for analysis.



In [ ]:
# Install missing packages. This cell does nothing if they are already installed.
import importlib.util
import subprocess
import sys

packages = ["pandas", "numpy", "requests", "meteostat", "matplotlib", "seaborn"]
missing = [p for p in packages if importlib.util.find_spec(p) is None]

if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
    print("Installed:", missing)
else:
    print("All packages are already installed.")



Installed: ['meteostat']


In [ ]:
import os
import warnings

import matplotlib.pyplot as plt
import meteostat as ms
import numpy as np
import pandas as pd
import requests
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)

# API keys: environment variables are safer than writing keys in the notebook.
OPENAQ_API_KEY = os.getenv("OPENAQ_API_KEY", "9da770ae15cb8ebb36b84891d96436f03579775b657196f38c7ea35cbea7954c")
PURPLEAIR_READ_KEY = os.getenv("PURPLEAIR_READ_KEY", "A5F0E1A3-B9D9-11F1-9E30-4201AC1DC129")

# Jakarta coordinates
WEST, SOUTH, EAST, NORTH = 106.689, -6.370, 106.978, -6.079
JAKARTA_LAT, JAKARTA_LON = -6.2088, 106.8456

START_DATE = "2024-01-01 00:00:00+07:00"
END_DATE = "2024-02-01 00:00:00+07:00"

START_UTC = pd.Timestamp(START_DATE).tz_convert("UTC")
END_UTC = pd.Timestamp(END_DATE).tz_convert("UTC")

print("Period:", START_UTC, "to", END_UTC)



Period: 2023-12-31 17:00:00+00:00 to 2024-01-31 17:00:00+00:00


## 1. Small API helper

The same function is used for both APIs. It returns `None` when an API cannot
be reached, so one unavailable source does not stop the entire notebook.



In [ ]:
def get_api_data(url, headers=None, params=None):
    try:
        response = requests.get(url, headers=headers, params=params, timeout=30)
        response.raise_for_status()
        return response.json()
    except requests.RequestException as error:
        print(f"API request failed: {url}")
        print(error)
        return None


def key_is_ready(key):
    return bool(key) and not key.startswith("PASTE_")



## 2. Download OpenAQ PM2.5

First, find fixed PM2.5 monitors inside Jakarta. Then download the hourly
measurements for every sensor found. The `X-API-Key` header is required by
OpenAQ.



In [ ]:
def download_openaq():
    if not key_is_ready(OPENAQ_API_KEY):
        print("OpenAQ key is missing.")
        return pd.DataFrame()

    headers = {"X-API-Key": OPENAQ_API_KEY}
    location_url = "https://api.openaq.org/v3/locations"
    location_params = {
        "bbox": f"{WEST},{SOUTH},{EAST},{NORTH}",
        "parameters_id": 2,
        "limit": 1000,
    }

    result = get_api_data(location_url, headers, location_params)
    if result is None:
        return pd.DataFrame()

    sensors = []
    for location in result.get("results", []):
        if location.get("isMobile") is True:
            continue

        coordinates = location.get("coordinates") or {}
        for sensor in location.get("sensors", []):
            parameter = sensor.get("parameter") or {}
            if parameter.get("id") == 2:
                sensors.append(
                    {
                        "sensor_id": sensor["id"],
                        "latitude": coordinates.get("latitude"),
                        "longitude": coordinates.get("longitude"),
                    }
                )

    sensors = pd.DataFrame(sensors).drop_duplicates()
    print("OpenAQ sensors:", len(sensors))

    rows = []
    for sensor in sensors.itertuples(index=False):
        url = f"https://api.openaq.org/v3/sensors/{sensor.sensor_id}/hours"
        params = {
            "datetime_from": START_UTC.isoformat(),
            "datetime_to": END_UTC.isoformat(),
            "limit": 1000,
        }
        result = get_api_data(url, headers, params)

        if result is None:
            continue

        for item in result.get("results", []):
            period = item.get("period") or {}
            start = period.get("datetimeFrom") or {}
            rows.append(
                {
                    "time": start.get("utc"),
                    "source": "OpenAQ",
                    "site_id": f"OpenAQ_{sensor.sensor_id}",
                    "latitude": sensor.latitude,
                    "longitude": sensor.longitude,
                    "pm25": item.get("value"),
                }
            )

    return pd.DataFrame(rows)


openaq_raw = download_openaq()
print("OpenAQ rows:", len(openaq_raw))
display(openaq_raw.head())



OpenAQ sensors: 20
OpenAQ rows: 3990


,time,source,site_id,latitude,longitude,pm25
0,2023-12-31T17:00:00Z,OpenAQ,OpenAQ_5077558,-6.236704,106.793244,133.0
1,2023-12-31T18:00:00Z,OpenAQ,OpenAQ_5077558,-6.236704,106.793244,122.0
2,2023-12-31T19:00:00Z,OpenAQ,OpenAQ_5077558,-6.236704,106.793244,73.0
3,2023-12-31T20:00:00Z,OpenAQ,OpenAQ_5077558,-6.236704,106.793244,59.0
4,2023-12-31T21:00:00Z,OpenAQ,OpenAQ_5077558,-6.236704,106.793244,58.0


## 3. Download PurpleAir PM2.5

PurpleAir may be blocked on some networks. If that happens, this cell prints
the connection error and returns an empty DataFrame instead of stopping the
notebook. The A and B particle channels are kept for a simple agreement check.



In [ ]:
def purpleair_dataframe(result):
    if result is None:
        return pd.DataFrame()
    return pd.DataFrame(result.get("data", []), columns=result.get("fields", []))


def download_purpleair():
    if not key_is_ready(PURPLEAIR_READ_KEY):
        print("PurpleAir key is missing.")
        return pd.DataFrame()

    headers = {"X-API-Key": PURPLEAIR_READ_KEY}
    sensor_url = "https://api.purpleair.com/v1/sensors"
    sensor_params = {
        "fields": "name,latitude,longitude,location_type",
        "location_type": 0,
        "nwlat": NORTH,
        "nwlng": WEST,
        "selat": SOUTH,
        "selng": EAST,
    }

    result = get_api_data(sensor_url, headers, sensor_params)
    sensors = purpleair_dataframe(result)
    print("PurpleAir sensors:", len(sensors))

    all_rows = []
    for sensor in sensors.itertuples(index=False):
        # One-hour averages for the selected month.
        history_url = (
            f"https://api.purpleair.com/v1/sensors/"
            f"{sensor.sensor_index}/history"
        )
        history_params = {
            "start_timestamp": int(START_UTC.timestamp()),
            "end_timestamp": int(END_UTC.timestamp()),
            "average": 60,
            "fields": "pm2.5_cf_1_a,pm2.5_cf_1_b",
        }

        result = get_api_data(history_url, headers, history_params)
        history = purpleair_dataframe(result)

        if history.empty:
            continue

        history["time"] = pd.to_datetime(history["time_stamp"], unit="s", utc=True)
        history["source"] = "PurpleAir"
        history["site_id"] = f"PurpleAir_{sensor.sensor_index}"
        history["latitude"] = sensor.latitude
        history["longitude"] = sensor.longitude
        all_rows.append(history)

    if not all_rows:
        return pd.DataFrame()

    return pd.concat(all_rows, ignore_index=True)


purpleair_raw = download_purpleair()
print("PurpleAir rows:", len(purpleair_raw))
display(purpleair_raw.head())



PurpleAir sensors: 2
PurpleAir rows: 743


,time_stamp,pm2.5_cf_1_a,pm2.5_cf_1_b,time,source,site_id,latitude,longitude
0,1706666400,43.0,30.7,2024-01-31 02:00:00+00:00,PurpleAir,PurpleAir_55391,-6.124591,106.85121
1,1706342400,32.0,22.2,2024-01-27 08:00:00+00:00,PurpleAir,PurpleAir_55391,-6.124591,106.85121
2,1706371200,56.0,40.0,2024-01-27 16:00:00+00:00,PurpleAir,PurpleAir_55391,-6.124591,106.85121
3,1706360400,69.4,49.0,2024-01-27 13:00:00+00:00,PurpleAir,PurpleAir_55391,-6.124591,106.85121
4,1706353200,45.2,31.7,2024-01-27 11:00:00+00:00,PurpleAir,PurpleAir_55391,-6.124591,106.85121


## 4. Basic cleaning

The cleaning rules are intentionally simple:

- remove duplicate rows;
- convert timestamps to UTC;
- accept PM2.5 values from 0 to 1000 µg/m³;
- for PurpleAir, remove rows where channels A and B differ too much;
- calculate hourly means;
- interpolate gaps of at most two hours.



In [ ]:
def clean_openaq(data):
    if data.empty:
        return data

    clean = data.drop_duplicates().copy()
    clean["time"] = pd.to_datetime(clean["time"], utc=True, errors="coerce")
    clean["pm25"] = pd.to_numeric(clean["pm25"], errors="coerce")
    clean.loc[~clean["pm25"].between(0, 1000), "pm25"] = np.nan
    return clean


def clean_purpleair(data):
    if data.empty:
        return pd.DataFrame()

    clean = data.drop_duplicates().copy()
    clean["time"] = pd.to_datetime(clean["time"], utc=True, errors="coerce")
    clean["pm2.5_cf_1_a"] = pd.to_numeric(clean["pm2.5_cf_1_a"], errors="coerce")
    clean["pm2.5_cf_1_b"] = pd.to_numeric(clean["pm2.5_cf_1_b"], errors="coerce")

    clean["pm25"] = clean[["pm2.5_cf_1_a", "pm2.5_cf_1_b"]].mean(axis=1)
    difference = (clean["pm2.5_cf_1_a"] - clean["pm2.5_cf_1_b"]).abs()
    allowed_difference = np.maximum(10, clean["pm25"] * 0.50)

    clean.loc[difference > allowed_difference, "pm25"] = np.nan
    clean.loc[~clean["pm25"].between(0, 1000), "pm25"] = np.nan

    return clean[["time", "source", "site_id", "latitude", "longitude", "pm25"]]


openaq_clean = clean_openaq(openaq_raw)
purpleair_clean = clean_purpleair(purpleair_raw)

air_data = pd.concat([openaq_clean, purpleair_clean], ignore_index=True)
air_data = air_data.dropna(subset=["time", "site_id", "latitude", "longitude"])

# Hourly mean for each sensor.
air_hourly = (
    air_data.set_index("time")
    .groupby(["source", "site_id", "latitude", "longitude"])["pm25"]
    .resample("1h")
    .mean()
    .reset_index()
)

# Fill only short gaps within the same sensor.
air_hourly["pm25_imputed"] = False
for site_id, indexes in air_hourly.groupby("site_id").groups.items():
    indexes = air_hourly.loc[indexes].sort_values("time").index
    was_missing = air_hourly.loc[indexes, "pm25"].isna()
    air_hourly.loc[indexes, "pm25"] = (
        air_hourly.loc[indexes, "pm25"]
        .interpolate(limit=2, limit_area="inside")
        .values
    )
    air_hourly.loc[indexes, "pm25_imputed"] = (
        was_missing & air_hourly.loc[indexes, "pm25"].notna()
    ).values

print("Clean hourly air-quality rows:", len(air_hourly))
display(air_hourly.head())



## 5. Download and clean weather data

Five nearby weather stations are requested from Meteostat. Weather variables
outside simple physical ranges are replaced with missing values. Small gaps
are interpolated, and remaining gaps are filled with each station's median.



In [ ]:
def download_weather():
    point = ms.Point(JAKARTA_LAT, JAKARTA_LON)
    station_query = ms.stations.nearby(point, limit=5).fetch()

    start = START_UTC.tz_localize(None).to_pydatetime()
    end = END_UTC.tz_localize(None).to_pydatetime()
    weather = ms.hourly(
        station_query,
        start,
        end,
        timezone="UTC",
    ).fetch().reset_index()

    stations = station_query.reset_index().rename(
        columns={
            "id": "station_id",
            "lat": "station_latitude",
            "lon": "station_longitude",
        }
    )

    weather = weather.rename(
        columns={
            "station": "station_id",
            "time": "time",
            "temp": "temperature",
            "rhum": "humidity",
            "prcp": "rain",
            "wspd": "wind_speed",
            "pres": "pressure",
        }
    )

    weather["station_id"] = weather["station_id"].astype(str)
    stations["station_id"] = stations["station_id"].astype(str)
    weather["time"] = pd.to_datetime(weather["time"], utc=True)

    return stations, weather


weather_stations, weather = download_weather()

# Simple physical limits.
limits = {
    "temperature": (-30, 60),
    "humidity": (0, 100),
    "rain": (0, 500),
    "wind_speed": (0, 250),
    "pressure": (850, 1100),
}

weather_columns = []
for column, (minimum, maximum) in limits.items():
    if column in weather.columns:
        weather_columns.append(column)
        weather[column] = pd.to_numeric(weather[column], errors="coerce")
        weather.loc[~weather[column].between(minimum, maximum), column] = np.nan

for station_id, indexes in weather.groupby("station_id").groups.items():
    indexes = weather.loc[indexes].sort_values("time").index
    weather.loc[indexes, weather_columns] = (
        weather.loc[indexes, weather_columns]
        .interpolate(limit=3, limit_area="inside")
        .fillna(weather.loc[indexes, weather_columns].median())
        .values
    )

print("Weather stations:", len(weather_stations))
display(weather.head())



## 6. Match each air sensor to the nearest weather station

The Haversine formula calculates distance between two latitude/longitude
points. Each air sensor receives the ID of its nearest weather station.



In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    radius = 6371
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    difference_lat = lat2 - lat1
    difference_lon = lon2 - lon1
    a = (
        np.sin(difference_lat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(difference_lon / 2) ** 2
    )
    return 2 * radius * np.arcsin(np.sqrt(a))


sites = air_hourly[["site_id", "latitude", "longitude"]].drop_duplicates()
links = []

for site in sites.itertuples(index=False):
    distances = haversine(
        site.latitude,
        site.longitude,
        weather_stations["station_latitude"].to_numpy(),
        weather_stations["station_longitude"].to_numpy(),
    )
    nearest = distances.argmin()
    links.append(
        {
            "site_id": site.site_id,
            "station_id": weather_stations.iloc[nearest]["station_id"],
            "distance_to_weather_km": round(float(distances[nearest]), 2),
        }
    )

links = pd.DataFrame(links)
display(links)



## 7. Combine the datasets

First join the spatial link by `site_id`, then join the weather values by
station ID and UTC hour. This creates the composite spatiotemporal linkage.



In [ ]:
integrated = air_hourly.merge(links, on="site_id", how="left")

weather_keep = ["station_id", "time"] + weather_columns
integrated = integrated.merge(
    weather[weather_keep],
    on=["station_id", "time"],
    how="left",
)

integrated["time_jakarta"] = integrated["time"].dt.tz_convert("Asia/Jakarta")
integrated["hour"] = integrated["time_jakarta"].dt.hour
integrated["day_of_week"] = integrated["time_jakarta"].dt.dayofweek

print("Integrated shape:", integrated.shape)
display(integrated.head())



## 8. Remove highly correlated weather columns

If two weather variables have an absolute correlation above 0.95, the second
one is removed. PM2.5 is not included in this decision.



In [ ]:
correlation = integrated[weather_columns].corr().abs()
upper = correlation.where(np.triu(np.ones(correlation.shape), k=1).astype(bool))

drop_columns = [
    column for column in upper.columns if any(upper[column] > 0.95)
]

print("Columns removed:", drop_columns)

analysis_ready = integrated.drop(columns=drop_columns)
analysis_ready = analysis_ready.drop_duplicates(
    subset=["source", "site_id", "time"]
)
analysis_ready = analysis_ready.sort_values(["time", "source", "site_id"])
analysis_ready = analysis_ready.reset_index(drop=True)

print("Final shape:", analysis_ready.shape)
display(analysis_ready.head())



## 9. Quick check and save



In [ ]:
plt.figure(figsize=(12, 5))
sns.lineplot(
    data=analysis_ready,
    x="time_jakarta",
    y="pm25",
    hue="source",
    estimator="mean",
    errorbar=None,
)
plt.title("Hourly PM2.5 in Jakarta")
plt.xlabel("Jakarta time")
plt.ylabel("PM2.5 (µg/m³)")
plt.tight_layout()
plt.show()

analysis_ready.to_csv("airQuality_clean.csv", index=False)
print("Saved: airQuality_clean.csv")

analysis_ready
